In [ ]:
#@title 🚀 1. 指挥中心 (Global Configuration)
import os, requests, shutil
from pathlib import Path
from google.colab import drive, files

# --- 1.1 基础环境 ---
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

!pip install --quiet --upgrade pybiolib
import biolib
biolib.login()

# --- 1.2 用户输入参数 ---
#@markdown ### 🧬 抗原与抗体设置
pdb_id = "3IU3" #@param {type:"string"}
target_chains = "K" #@param {type:"string"}
hotspot_res = "" #@param {type:"string"}
design_loops = "[L1:8-13,L2:7,L3:9-11,H1:7,H2:6,H3:5-13]" #@param {type:"string"}

#@markdown ### ⚙️ 运行规模
num_designs = 32 #@param {type:"integer"}
num_seqs_per_struct = 4 #@param {type:"integer"}

# --- 1.3 自动路径计算 ---
pdb_id = pdb_id.strip().upper()
# 所有过程文件存放在 Drive，方便断线后找回
project_dir = f"/content/drive/MyDrive/AntibodyDesign_{pdb_id}"
os.makedirs(project_dir, exist_ok=True)

antigen_final_pdb = os.path.join(project_dir, f"antigen_{pdb_id}_T.pdb")
framework_pdb = os.path.join(project_dir, "framework.pdb")
framework_url = "https://raw.githubusercontent.com/mhoie/workshop/refs/heads/main/hu-4D5-8_Fv.pdb"

print(f"✅ 项目目录: {project_dir}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 4.8 MB/s eta 0:00:00
Opening authorization page at: https://biolib.com/sign-in/request/notebook/?token=yjjUGhDJsZnwZtNN
If your browser does not open automatically, click on the link above.


<IPython.core.display.Javascript object>

Successfully signed in!
✅ 项目目录: /content/drive/MyDrive/AntibodyDesign_3IU3


In [ ]:
#@title 🛠️ 2. 自动化数据准备 (Antigen & Framework Prep)
#@markdown 自动处理下载、多链合并、重编号、HLT 格式化，并彻底去杂。

def process_antigen_hlt(pdb_id, chains, output_path, gap=50):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    resp = requests.get(url)
    if resp.status_code != 200: raise ValueError(f"PDB {pdb_id} 下载失败")

    lines = resp.text.splitlines()
    selected_chains = [c.strip().upper() for c in chains.split(',')]
    output_lines = []
    g_atom_idx, g_res_idx = 1, 1

    for chain in selected_chains:
        # 仅保留 ATOM，自动剔除 HETATM, 氢原子和水
        chain_atoms = [l for l in lines if l.startswith("ATOM") and l[21].strip() == chain]
        if not chain_atoms: continue

        last_old_res = None
        for line in chain_atoms:
            old_res = line[22:26].strip()
            # 检测残基编号变化
            if last_old_res is not None and old_res != last_old_res:
                g_res_idx += 1
            last_old_res = old_res

            # 构建标准的 HLT 格式行 (Chain T, B-factor 0.00)
            new_line = (f"{line[:6]}{g_atom_idx:>5}{line[11:21]}T{g_res_idx:>4}"
                        f"{line[26:60]}  0.00{line[66:]}")
            output_lines.append(new_line)
            g_atom_idx += 1
        # 链间插入 Gap 防止模型误判为共价连接
        g_res_idx += gap

    with open(output_path, 'w') as f:
        f.write("\n".join(output_lines) + "\nEND\n")
    print(f"✅ 目标抗原已生成: {output_path}")

# 执行处理逻辑
process_antigen_hlt(pdb_id, target_chains, antigen_final_pdb)

# 下载 Framework
if not os.path.exists(framework_pdb):
    !wget -q "$framework_url" -O "$framework_pdb"
    print(f"✅ 框架文件已就绪")

print(f"🚀 数据准备完毕，准备进入计算阶段。")

✅ 目标抗原已生成: /content/drive/MyDrive/AntibodyDesign_3IU3/antigen_3IU3_T.pdb
✅ 框架文件已就绪
🚀 数据准备完毕，准备进入计算阶段。


In [ ]:
#@title 🧬 3. 运行设计流水线 (~25-30 分钟)
#@markdown 依次运行 RFdiffusion, ProteinMPNN 和 RosettaFold2。

# 1. RFdiffusion
print("1/3 🌊 正在运行 RFdiffusion...")
app_rfdiff = biolib.load('BioITWorkshop/RFDiffusionAntibody')
job_rfdiff = app_rfdiff.run(
    target_pdb=antigen_final_pdb,
    framework_pdb=framework_pdb,
    hotspot_res=hotspot_res,
    design_loops=design_loops,
    num_designs=num_designs,
)
rfdiff_local = "/content/rfdiffusion"
job_rfdiff.save_files(rfdiff_local, path_filter=lambda p: p.endswith('.pdb') and 'traj' not in p)

# 2. ProteinMPNN
print("2/3 🎨 正在运行 ProteinMPNN...")
app_mpnn = biolib.load('BioITWorkshop/ProteinMPNN')
job_mpnn = app_mpnn.run(
    pdb=rfdiff_local,
    num_seqs_per_struct=num_seqs_per_struct
)
mpnn_local = "/content/proteinmpnn"
job_mpnn.save_files(mpnn_local)

# 3. RF2 验证
print("3/3 🔬 正在运行 RF2 验证...")
app_rf2 = biolib.load('BioITWorkshop/RF2Antibody')
job_rf2 = app_rf2.run(
    pdb=mpnn_local,
    num_recycles=3,
)
rf2_local = "/content/rosettafold2"
job_rf2.save_files(rf2_local)

# 将最终结果同步到 Drive 备份
shutil.copytree(rf2_local, os.path.join(project_dir, "final_designs"), dirs_exist_ok=True)
print(f"✨ 流程全部结束！结果已同步至 Google Drive 项目文件夹。")

1/3 🌊 正在运行 RFdiffusion...


NameError: name 'biolib' is not defined

In [ ]:
#@title 📦 4. 结果一键打包下载
#@markdown 运行此单元格会将所有经过 RF2 验证的 PDB 结果打包下载到你的本地。

zip_filename = f"{pdb_id}_Final_Designs.zip"
zip_path = f"/content/{zip_filename}"

# 压缩 rosettafold2 文件夹 (这是最终验证过的结构)
print(f"正在打包最终结果...")
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', rf2_local)

# 触发浏览器下载
print(f"正在准备下载: {zip_filename}")
files.download(zip_path)

print("✅ 下载指令已发送。")